# Notebook 03: Structured Outputs with the Responses API

This notebook shows how to move from free-form prose to machine-usable JSON using a strict schema.

You will:
- define a JSON schema for a meeting summary
- extract structured data with `create_json_response(...)`
- turn nested action items into a pandas table
- reuse the same pattern on a second example


## What you are building

In production, structured outputs are usually more valuable than a nice paragraph because they can feed tables, filters, automations, and downstream tools.

The helper in `utils.responses_api.create_json_response(...)` wraps the strict JSON schema format for the Responses API so the notebook code stays readable.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / '.env')

from utils import DEFAULT_MODEL, create_json_response
from utils.models import choose_default_model

print('Project root:', PROJECT_ROOT)
print('DEFAULT_MODEL:', DEFAULT_MODEL)


In [ ]:
# Optional per-notebook model preference
PREFERENCE = 'quality'
MODEL_FOR_NOTEBOOK = choose_default_model(PREFERENCE)
print('Using model:', MODEL_FOR_NOTEBOOK)


## Example 1: Extract a meeting summary into JSON

We will ask the model to convert messy notes into a predictable object with:
- core account context
- overall deal status
- a short summary
- a list of concrete action items


In [ ]:
meeting_notes = """
Sales call notes

Attendees: Maya Chen from Northwind Health and Jordan from our team.
Northwind is evaluating our internal knowledge assistant for a 250-person operations group.
They like the retrieval quality but are concerned about rollout timing and employee training.
Budget is approved if the pilot works.
Target pilot date is April 15.
Jordan needs to send a revised proposal by next Tuesday.
Maya wants two customer references in healthcare.
We should schedule a technical workshop with their IT lead.
Overall tone was positive and the deal looks active.
""".strip()

MEETING_SCHEMA = {
    'type': 'object',
    'properties': {
        'contact_name': {'type': 'string'},
        'company': {'type': 'string'},
        'deal_status': {
            'type': 'string',
            'enum': ['discovery', 'active', 'at_risk', 'closed_won', 'closed_lost'],
        },
        'sentiment': {
            'type': 'string',
            'enum': ['positive', 'neutral', 'negative'],
        },
        'pilot_date': {'type': 'string'},
        'summary': {'type': 'string'},
        'action_items': {
            'type': 'array',
            'items': {
                'type': 'object',
                'properties': {
                    'owner': {'type': 'string'},
                    'task': {'type': 'string'},
                    'deadline': {'type': 'string'},
                },
                'required': ['owner', 'task', 'deadline'],
                'additionalProperties': False,
            },
        },
    },
    'required': [
        'contact_name',
        'company',
        'deal_status',
        'sentiment',
        'pilot_date',
        'summary',
        'action_items',
    ],
    'additionalProperties': False,
}


In [ ]:
structured_meeting = create_json_response(
    meeting_notes,
    model=MODEL_FOR_NOTEBOOK,
    schema_name='sales_meeting_summary',
    schema=MEETING_SCHEMA,
    instructions=(
        'Extract the meeting notes into the schema exactly. '
        'Use concise language. If a deadline is implied but not precise, say "unspecified".'
    ),
)

print(json.dumps(structured_meeting, indent=2))


## Turn nested items into a table

Once the output is structured, moving into pandas is straightforward.

In [ ]:
action_items_df = pd.DataFrame(structured_meeting['action_items'])
action_items_df.insert(0, 'company', structured_meeting['company'])
action_items_df.insert(1, 'contact_name', structured_meeting['contact_name'])
action_items_df


## Example 2: Support ticket triage

The same helper works for classification and routing problems, not just meeting summaries.

In [ ]:
ticket_text = """
Customer email:
Hi team,

Our finance users cannot export invoices from the dashboard after yesterday's update.
This is blocking end-of-week reconciliation for about 12 people.
Please call me if needed. We need this fixed today.

Thanks,
Alicia
""".strip()

TICKET_SCHEMA = {
    'type': 'object',
    'properties': {
        'request_type': {'type': 'string'},
        'priority': {'type': 'string', 'enum': ['low', 'medium', 'high', 'urgent']},
        'team': {'type': 'string'},
        'customer_impact': {'type': 'string'},
        'suggested_next_step': {'type': 'string'},
    },
    'required': [
        'request_type',
        'priority',
        'team',
        'customer_impact',
        'suggested_next_step',
    ],
    'additionalProperties': False,
}

ticket_result = create_json_response(
    ticket_text,
    model=MODEL_FOR_NOTEBOOK,
    schema_name='support_ticket_triage',
    schema=TICKET_SCHEMA,
    instructions='Classify the ticket for an internal support queue. Be concise and practical.',
)

ticket_result


## Why this notebook matters

Structured outputs are the bridge between prompt demos and real systems. Once the model returns data that fits a schema, you can validate it, store it, filter it, and hand it off to tools or workflows without brittle string parsing.